# Planning Algorithms

Notebook này trình bày quy trình thực hành trực tiếp cho nhóm **Planning Algorithms** trên Grid-world 8x8.

Planning Algorithms dùng khi biết model của MDP. Agent biết $P(s' \mid s,a)$ và $r(s,a,s')$, do đó có thể tính value function/policy trực tiếp bằng Bellman equations thay vì học bằng trial-and-error.

Các thuật toán:

1. Policy Evaluation
2. Policy Iteration
3. Value Iteration
4. Linear Programming Planner

## Mục lục

- Setup imports
- Thiết kế Grid-world 8x8
- Policy Evaluation
- Policy Iteration
- Value Iteration
- Linear Programming Planner
- Đánh giá và so sánh Planning models
- Tổng kết Planning

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

In [ ]:
import json
from pathlib import Path
from time import perf_counter

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from envs.planning_grid_world import PlanningGridWorld
from agents.planning import (
    PolicyEvaluation,
    PolicyIteration,
    ValueIteration,
    LinearProgrammingPlanner,
)
from utils.metrics import mean_squared_error, max_abs_error, policy_agreement
from utils.visualization import (
    plot_value_heatmap,
    plot_policy_arrows,
    plot_bellman_residual,
    plot_comparison_bar,
)

NOTEBOOK_FIGURE_DIR = PROJECT_ROOT / "report" / "figures" / "notebook_demos" / "planning"
NOTEBOOK_VERBOSE = 2
NOTEBOOK_LOG_INTERVAL = 5

In [ ]:
def show_metrics(metrics, keys=None):
    if metrics is None:
        print("No metrics available.")
        return
    selected = metrics if keys is None else {key: metrics.get(key) for key in keys}
    for key, value in selected.items():
        print(f"{key}: {value}")


def load_json(path):
    path = Path(path)
    if not path.exists():
        print(f"Missing file: {path}")
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def show_png(path, width=720):
    path = Path(path)
    if not path.exists():
        print(f"Missing figure: {path}")
        return
    display(Image(filename=str(path), width=width))

## Thiết kế Grid-world 8x8

Grid-world là môi trường rời rạc. State là tọa độ ô `(row, col)`. Action là các hành động di chuyển như `up`, `down`, `left`, `right`.

- **Goal** là trạng thái kết thúc tốt.
- **Trap** là trạng thái kết thúc xấu.
- **Wall** là ô không thể đi qua.
- **Reward** gồm step reward, goal reward, trap reward và wall/boundary penalty theo config hiện tại.
- **Terminal states** gồm Goal và Trap.

Grid-world có thể được mô hình hóa như một MDP:

$$M=(S,A,P,r,\gamma)$$

Planning dùng model đầy đủ qua `get_transitions(state, action)`. Learning chỉ quan sát sample qua `reset()` và `step(action)` trong training.

In [ ]:
planning_env = PlanningGridWorld()
planning_env

In [ ]:
print("Grid size:", planning_env.grid_size)
print("Start state:", planning_env.start_state)
print("Goal states:", planning_env.goal_states)
print("Trap states:", planning_env.trap_states)
print("Wall states:", planning_env.wall_states)
print("Actions:", planning_env.actions)
print("Reward config:", planning_env.reward_config)
print("Gamma:", planning_env.gamma)
print("Number of states:", planning_env.num_states())

In [ ]:
# TODO: Nếu đã có helper visualize layout riêng, gọi tại đây.
# Ví dụ: plot_grid_world_layout(planning_env, title="Grid-world 8x8")
# Hiện project có thể visualize value/policy sau khi chạy thuật toán.

## Xây dựng và chạy các thuật toán Planning

Các cell dưới đây tạo model, gọi `.run()`, lấy metrics từ model vừa chạy, đánh giá trực tiếp và visualize kết quả bằng helper trong `utils.visualization`.

## Policy Evaluation

**Mục tiêu:** tính $V^\pi$ cho một policy cố định.

Bellman Expectation Equation:

$$V^\pi(s)=\sum_a \pi(a\mid s)\sum_{s'}P(s'\mid s,a)[r(s,a,s')+\gamma V^\pi(s')]$$

**Cơ chế:** chọn policy cố định, khởi tạo $V(s)=0$, lặp Bellman expectation backup, dừng khi residual nhỏ hơn threshold.

Trong project, nếu không truyền `policy`, `PolicyEvaluation` dùng uniform random policy mặc định. Đây là baseline phù hợp cho TD(0)/TD(lambda).

In [ ]:
policy_evaluator = PolicyEvaluation(
    env=planning_env,
    policy=None,  # dùng uniform random policy mặc định trong class
    theta=1e-6,
    max_iterations=1000,
    verbose=NOTEBOOK_VERBOSE,
    log_interval=NOTEBOOK_LOG_INTERVAL,
)

start_time = perf_counter()
pe_result = policy_evaluator.run()
pe_runtime = perf_counter() - start_time

pe_values = policy_evaluator.get_value_function()
pe_policy = policy_evaluator.get_policy()
pe_metrics = policy_evaluator.get_metrics()
pe_metrics["notebook_runtime_sec"] = pe_runtime

In [ ]:
show_metrics(pe_metrics, keys=[
    "status",
    "iterations",
    "final_bellman_residual",
    "bellman_backups",
    "notebook_runtime_sec",
])

# Đánh giá trực tiếp residual cuối từ metrics của model vừa chạy.
print("Residual < theta:", pe_metrics["final_bellman_residual"] < pe_metrics["theta"])

In [ ]:
pe_value_path = NOTEBOOK_FIGURE_DIR / "policy_evaluation_value_heatmap.png"
pe_residual_path = NOTEBOOK_FIGURE_DIR / "policy_evaluation_residual.png"

plot_value_heatmap(pe_values, planning_env, "Policy Evaluation - V^pi", pe_value_path)
plot_bellman_residual(pe_metrics["bellman_residuals"], "Policy Evaluation Residual", pe_residual_path)

show_png(pe_value_path)
show_png(pe_residual_path)

**Nhận xét cần điền sau khi chạy:**

- Thuật toán hội tụ sau bao nhiêu iteration?
- Residual cuối có nhỏ hơn threshold không?
- Các state gần Goal/Trap có value như kỳ vọng không?

## Policy Iteration

**Mục tiêu:** tìm policy tối ưu bằng policy evaluation + policy improvement.

Policy improvement:

$$\pi_{new}(s)=\arg\max_a\sum_{s'}P(s'\mid s,a)[r(s,a,s')+\gamma V^\pi(s')]$$

**Cơ chế:** khởi tạo policy, evaluate policy, improve policy, lặp đến khi policy stable.

In [ ]:
policy_iteration = PolicyIteration(
    env=planning_env,
    theta=1e-6,
    max_iterations=1000,
    verbose=NOTEBOOK_VERBOSE,
    log_interval=NOTEBOOK_LOG_INTERVAL,
)

start_time = perf_counter()
pi_result = policy_iteration.run()
pi_runtime = perf_counter() - start_time

pi_values = policy_iteration.get_value_function()
pi_policy = policy_iteration.get_policy()
pi_metrics = policy_iteration.get_metrics()
pi_metrics["notebook_runtime_sec"] = pi_runtime

In [ ]:
show_metrics(pi_metrics, keys=[
    "status",
    "policy_improvement_iterations",
    "total_policy_evaluation_iterations",
    "final_bellman_residual",
    "bellman_backups",
    "notebook_runtime_sec",
])

print("Policy stable:", pi_metrics.get("policy_stable"))

In [ ]:
pi_value_path = NOTEBOOK_FIGURE_DIR / "policy_iteration_value_heatmap.png"
pi_policy_path = NOTEBOOK_FIGURE_DIR / "policy_iteration_policy.png"
pi_changes_path = NOTEBOOK_FIGURE_DIR / "policy_iteration_policy_changes.png"

plot_value_heatmap(pi_values, planning_env, "Policy Iteration - V*", pi_value_path)
plot_policy_arrows(pi_policy, planning_env, "Policy Iteration - Greedy Policy", pi_policy_path)
plot_bellman_residual(pi_metrics["policy_changes_per_iteration"], "Policy Iteration Policy Changes", pi_changes_path)

show_png(pi_value_path)
show_png(pi_policy_path)
show_png(pi_changes_path)

**Nhận xét cần điền sau khi chạy:**

- Policy stable sau bao nhiêu vòng improvement?
- Số policy changes giảm như thế nào?
- Policy cuối có hợp lý trên Grid-world không?

## Value Iteration

**Mục tiêu:** tìm $V^*$ trực tiếp bằng Bellman Optimality Backup.

$$V_{k+1}(s)=\max_a\sum_{s'}P(s'\mid s,a)[r(s,a,s')+\gamma V_k(s')]$$

**Cơ chế:** khởi tạo value, lặp Bellman optimality backup, sau hội tụ trích xuất greedy policy.

Value Iteration đóng vai trò baseline cho SARSA/Q-learning và để so sánh với Policy Iteration/LP.

In [ ]:
value_iteration = ValueIteration(
    env=planning_env,
    theta=1e-6,
    max_iterations=1000,
    verbose=NOTEBOOK_VERBOSE,
    log_interval=NOTEBOOK_LOG_INTERVAL,
)

start_time = perf_counter()
vi_result = value_iteration.run()
vi_runtime = perf_counter() - start_time

vi_values = value_iteration.get_value_function()
vi_policy = value_iteration.get_policy()
vi_metrics = value_iteration.get_metrics()
vi_metrics["notebook_runtime_sec"] = vi_runtime

In [ ]:
show_metrics(vi_metrics, keys=[
    "status",
    "iterations",
    "final_bellman_residual",
    "bellman_backups",
    "notebook_runtime_sec",
])

print("Residual < theta:", vi_metrics["final_bellman_residual"] < vi_metrics["theta"])

In [ ]:
vi_value_path = NOTEBOOK_FIGURE_DIR / "value_iteration_value_heatmap.png"
vi_policy_path = NOTEBOOK_FIGURE_DIR / "value_iteration_policy.png"
vi_residual_path = NOTEBOOK_FIGURE_DIR / "value_iteration_residual.png"

plot_value_heatmap(vi_values, planning_env, "Value Iteration - V*", vi_value_path)
plot_policy_arrows(vi_policy, planning_env, "Value Iteration - Optimal Policy", vi_policy_path)
plot_bellman_residual(vi_metrics["bellman_residuals"], "Value Iteration Residual", vi_residual_path)

show_png(vi_value_path)
show_png(vi_policy_path)
show_png(vi_residual_path)

**Nhận xét cần điền sau khi chạy:**

- Value Iteration hội tụ sau bao nhiêu iterations?
- Heatmap $V^*$ phản ánh Goal/Trap/Wall như thế nào?
- Policy greedy có đường đi hợp lý đến Goal không?

## Linear Programming Planner

**Mục tiêu:** giải MDP bằng tối ưu tuyến tính.

Ràng buộc Bellman optimality:

$$V(s)\ge r(s,a)+\gamma\sum_{s'}P(s'\mid s,a)V(s')$$

Objective:

$$\min_V \sum_s V(s)$$

**Cơ chế:** tạo biến $V(s)$, tạo constraints cho state-action, giải LP, trích xuất policy từ nghiệm.

In [ ]:
lp_planner = LinearProgrammingPlanner(
    env=planning_env,
    verbose=1,
)

start_time = perf_counter()
lp_result = lp_planner.run()
lp_runtime = perf_counter() - start_time

lp_values = lp_planner.get_value_function()
lp_policy = lp_planner.get_policy()
lp_metrics = lp_planner.get_metrics()
lp_metrics["notebook_runtime_sec"] = lp_runtime

In [ ]:
show_metrics(lp_metrics, keys=[
    "status",
    "number_of_variables",
    "number_of_constraints",
    "solver_iter",
    "solver_iterations",
    "notebook_runtime_sec",
])

In [ ]:
lp_value_path = NOTEBOOK_FIGURE_DIR / "linear_programming_value_heatmap.png"
lp_policy_path = NOTEBOOK_FIGURE_DIR / "linear_programming_policy.png"

plot_value_heatmap(lp_values, planning_env, "Linear Programming - V*", lp_value_path)
plot_policy_arrows(lp_policy, planning_env, "Linear Programming - Derived Policy", lp_policy_path)

show_png(lp_value_path)
show_png(lp_policy_path)

**Nhận xét cần điền sau khi chạy:**

- LP tạo bao nhiêu variables/constraints?
- Nghiệm LP gần Value Iteration ở mức nào?
- LP minh họa liên hệ giữa RL và Optimization ra sao?

## Đánh giá và so sánh Planning models

Cell dưới đây đánh giá trực tiếp từ các model vừa chạy, không phụ thuộc vào logs cũ.

In [ ]:
non_terminal_states = [state for state in planning_env.get_states() if not planning_env.is_terminal(state)]

planning_comparison = {
    "PI_vs_VI_inf_error": max_abs_error(pi_values, vi_values),
    "LP_vs_VI_inf_error": max_abs_error(lp_values, vi_values),
    "PI_vs_VI_mse": mean_squared_error(pi_values, vi_values),
    "LP_vs_VI_mse": mean_squared_error(lp_values, vi_values),
    "PI_vs_VI_policy_agreement": policy_agreement(pi_policy, vi_policy, non_terminal_states),
    "LP_vs_VI_policy_agreement": policy_agreement(lp_policy, vi_policy, non_terminal_states),
}

show_metrics(planning_comparison)

In [ ]:
comparison_error_path = NOTEBOOK_FIGURE_DIR / "planning_value_error_comparison.png"
comparison_agreement_path = NOTEBOOK_FIGURE_DIR / "planning_policy_agreement_comparison.png"

plot_comparison_bar(
    {
        "PI vs VI": planning_comparison["PI_vs_VI_inf_error"],
        "LP vs VI": planning_comparison["LP_vs_VI_inf_error"],
    },
    "Planning Value Error vs Value Iteration",
    comparison_error_path,
    ylabel="Infinity norm error",
)
plot_comparison_bar(
    {
        "PI vs VI": planning_comparison["PI_vs_VI_policy_agreement"],
        "LP vs VI": planning_comparison["LP_vs_VI_policy_agreement"],
    },
    "Planning Policy Agreement vs Value Iteration",
    comparison_agreement_path,
    ylabel="Agreement",
)

show_png(comparison_error_path)
show_png(comparison_agreement_path)

## Tổng kết Planning

Policy Evaluation là prediction/evaluation cho một policy cố định. Policy Iteration và Value Iteration tìm nghiệm tối ưu bằng Bellman equations. Linear Programming đưa bài toán MDP về optimization.

Trong notebook này, Value Iteration đóng vai trò baseline để so sánh PI và LP. Các kết quả cần nhận xét trong report gồm residual curves, value heatmap, policy arrows, agreement PI/LP với VI, runtime và Bellman backups.

TODO: Viết phần phân tích cuối cùng sau khi chạy toàn bộ notebook và chọn figures phù hợp cho báo cáo.